In [1]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)
print("Processor loaded successfully!")

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
print("Model loaded successfully!")

/venv/vlm-llava/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processor loaded successfully!


Loading weights: 100%|██████████| 686/686 [00:02<00:00, 249.56it/s, Materializing param=model.vision_tower.vision_model.pre_layrnorm.weight]                        


Model loaded successfully!


In [3]:
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch

VG_ROOT = Path("../data/visual_genome")
VG_OBJECTS_PATH = VG_ROOT / "VisualGenome_task" / "objects.json"
OUTPUT_PATH = "../results/infer/visual_genome/llava15/normal.json"
BATCH_SIZE = 16
MAX_NEW_TOKENS = 256
PROMPT = "Describe this image."

In [4]:
processor.tokenizer.padding_side = "left"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token


def batch_list(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def resolve_vg_image_path(vg_root: Path, image_id: int) -> Path:
    """
    Supports full Visual Genome structure:
      data/visual_genome/images/VG_100K/{id}.jpg
      data/visual_genome/images2/VG_100K_2/{id}.jpg
    """
    candidates = [
        vg_root / "images2" / "VG_100K_2" / f"{image_id}.jpg",
        vg_root / "images" / "VG_100K" / f"{image_id}.jpg",
        vg_root / f"{image_id}.jpg",  # optional small-folder fallback
    ]

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        f"Could not find image {image_id}.jpg. Tried:\n"
        + "\n".join(str(p) for p in candidates)
    )


def load_image(path: Path):
    return Image.open(path).convert("RGB")


def clean_output(text: str) -> str:
    text = text.strip()
    if "ASSISTANT:" in text:
        text = text.split("ASSISTANT:", 1)[-1].strip()
    return text


# -----------------------------
# Load first 100 Visual Genome image IDs
# -----------------------------
with open(VG_OBJECTS_PATH, "r", encoding="utf-8") as f:
    vg_objects = json.load(f)

vg_objects = vg_objects[:100]
image_ids = [int(obj["image_id"]) for obj in vg_objects]

print(f"Loaded {len(image_ids)} Visual Genome image IDs.")
print("First 5 IDs:", image_ids[:5])


# -----------------------------
# Run batch inference
# -----------------------------
results = []

hf_prompt = f"USER: <image>\n{PROMPT} ASSISTANT:"

for batch_ids in tqdm(list(batch_list(image_ids, BATCH_SIZE))):
    batch_images = []
    batch_prompts = []
    batch_paths = []
    valid_ids = []

    for image_id in batch_ids:
        image_path = resolve_vg_image_path(VG_ROOT, image_id)
        image = load_image(image_path)

        batch_images.append(image)
        batch_prompts.append(hf_prompt)
        batch_paths.append(str(image_path))
        valid_ids.append(image_id)

    inputs = processor(
        text=batch_prompts,
        images=batch_images,
        return_tensors="pt",
        padding=True
    )

    input_device = next(model.parameters()).device
    inputs = {
        k: v.to(input_device) if isinstance(v, torch.Tensor) else v
        for k, v in inputs.items()
    }

    input_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=processor.tokenizer.eos_token_id
        )

    generated_ids = output_ids[:, input_len:]

    outputs = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    for image_id, image_path, output in zip(valid_ids, batch_paths, outputs):
        caption = clean_output(output)

        results.append({
            "image_id": image_id,
            "path": image_path,
            "text": caption
        })

# -----------------------------
# Save CCEval-compatible output
# -----------------------------
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(results)} captions to {OUTPUT_PATH}")

Loaded 100 Visual Genome image IDs.
First 5 IDs: [1, 2, 3, 4, 5]


100%|██████████| 7/7 [01:13<00:00, 10.56s/it]


Saved 100 captions to ../results/infer/visual_genome/llava15/normal.json
